# Módulo 07 · Aula 02 — Design e Funcionalidades

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

## A dor da Aurora

> *"O gateway de pagamento avisa quando o boleto é pago. Só que o nosso 'aviso' é um script que pergunta a cada 5 minutos se tem novidade. Resultado: o cliente paga e espera até 5 minutos para o pedido liberar. E o gateway já reclamou que a gente consulta demais."*

E, na mesma reunião:

> *"O relatório de fechamento demora 40 segundos. Quando o financeiro clica, a tela fica branca e ele clica de novo. Aí são dois relatórios rodando, e a API para."*

> *"A tela de estoque consulta o banco a cada 2 segundos, para 30 pessoas ao mesmo tempo. É a mesma pergunta, 900 vezes por minuto."*

Quatro problemas, quatro ferramentas:

| Dor | Ferramenta |
|-----|-----------|
| Perguntar de 5 em 5 minutos | **Webhook** — eles avisam você |
| Requisição que demora 40s | **Background task** |
| A mesma pergunta 900×/min | **Cache** |
| Tela que precisa atualizar sozinha | **WebSocket** |

## O que você vai aprender aqui

| # | Tópico | Por que importa |
|---|--------|-----------------|
| 1 | **Webhooks** | 🔴 Receber é mais perigoso do que enviar |
| 2 | Assinatura HMAC | Provar que veio de quem diz |
| 3 | Idempotência | O mesmo evento vai chegar duas vezes |
| 4 | Upload de arquivos | 🔴 Onde moram três vulnerabilidades |
| 5 | Download e streaming | Não carregar 2 GB na memória |
| 6 | `BackgroundTasks` | E onde ele **não** basta |
| 7 | Cache | Chave, TTL, invalidação, estouro |
| 8 | WebSockets | Quando vale, e quando é exagero |

## ⚙️ Preparação

Usamos `fakeredis` para o cache — a **mesma API** do Redis de verdade, sem precisar de servidor. Trocar por `redis.Redis(...)` é mudar uma linha.

> ⚠️ Como no M05 com o `mongomock`: simulacro cobre o caminho comum e falha nas bordas. Ao final da seção de cache há uma lista do que o `fakeredis` **não** reproduz.

**Execute a célula abaixo antes de tudo.**

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Preparação do Módulo 07 · Aula 02
# ═══════════════════════════════════════════════════════════════
import hashlib
import hmac
import json
import os
import secrets
import shutil
import socket
import subprocess
import sys
import threading
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

warnings.filterwarnings("ignore")


def _garantir(pacote, importar=None):
    nome = importar or pacote
    try:
        __import__(nome)
        return True
    except ImportError:
        print(f"  instalando {pacote}...")
        subprocess.run([sys.executable, "-m", "pip", "install", pacote, "--quiet"],
                       check=False, capture_output=True)
        try:
            __import__(nome)
            return True
        except ImportError:
            print(f"  ⚠️ {pacote} indisponível")
            return False


for _p, _m in [("fastapi", "fastapi"), ("httpx", "httpx"),
               ("uvicorn[standard]", "uvicorn"),
               ("python-multipart", "multipart"),
               ("fakeredis", "fakeredis")]:
    _garantir(_p, _m)

import httpx
import uvicorn
from fastapi import FastAPI
from fastapi.testclient import TestClient

print(f"✅ httpx {httpx.__version__}")

TEM_REDIS_REAL = False
try:
    with socket.create_connection(("localhost", 6379), timeout=0.5):
        TEM_REDIS_REAL = True
except OSError:
    pass


def abrir_cache():
    """Redis de verdade se houver; senão, fakeredis.

    🔑 A API é idêntica — é esse o ponto. O código que usa o cache não
       sabe qual dos dois está por baixo.
    """
    if TEM_REDIS_REAL:
        import redis
        print("🟥 Redis real em localhost:6379")
        return redis.Redis(decode_responses=True)
    import fakeredis
    print("📦 fakeredis (em memória) — a mesma API, sem servidor")
    return fakeredis.FakeStrictRedis(decode_responses=True)


# ═══════════════════════════════════════════════════════════════
#  Servidor real numa thread (o mesmo da aula anterior)
# ═══════════════════════════════════════════════════════════════

def _porta_livre() -> int:
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


class Servico:
    def __init__(self, app: FastAPI, nome: str = "servico"):
        self.app, self.nome = app, nome
        self.porta = _porta_livre()
        self.url = f"http://127.0.0.1:{self.porta}"
        self._servidor = self._thread = None

    def iniciar(self, timeout: float = 15.0) -> "Servico":
        config = uvicorn.Config(self.app, host="127.0.0.1", port=self.porta,
                                log_level="critical", access_log=False)
        self._servidor = uvicorn.Server(config)
        self._thread = threading.Thread(target=self._servidor.run, daemon=True)
        self._thread.start()
        limite = time.monotonic() + timeout
        while time.monotonic() < limite:
            if self._servidor.started:
                print(f"🟢 {self.nome} no ar em {self.url}")
                return self
            time.sleep(0.05)
        raise RuntimeError(f"{self.nome} não subiu")

    def parar(self) -> None:
        if self._servidor is not None:
            self._servidor.should_exit = True
            self._thread.join(timeout=10)
            print(f"⚫ {self.nome} desligado")


# ═══════════════════════════════════════════════════════════════
#  Auxiliares
# ═══════════════════════════════════════════════════════════════

def req(cliente, metodo: str, caminho: str, mostrar_corpo=True, **kwargs):
    resposta = getattr(cliente, metodo.lower())(caminho, **kwargs)
    cor = {2: "✅", 3: "↪️", 4: "⚠️", 5: "🔴"}.get(resposta.status_code // 100, "  ")
    print(f"{cor} {metodo.upper():<7} {caminho:<40} → {resposta.status_code}")
    if mostrar_corpo:
        try:
            texto = json.dumps(resposta.json(), ensure_ascii=False, indent=2)
            linhas = texto.splitlines()
            for linha in linhas[:12]:
                print(f"   {linha}")
            if len(linhas) > 12:
                print(f"   ... (+{len(linhas) - 12} linhas)")
        except Exception:
            if resposta.text.strip():
                print(f"   {resposta.text[:180]}")
    print()
    return resposta


TRABALHO = Path("aula_07_02").resolve()
if TRABALHO.exists():
    shutil.rmtree(TRABALHO)
(TRABALHO / "recebidos").mkdir(parents=True)

print(f"📁 {TRABALHO}")
print("✅ `Servico`, `req()` e `abrir_cache()` prontos")

## 1. Webhooks — a inversão

Na aula anterior, **você perguntava**. Agora **eles avisam**.

```
POLLING                          WEBHOOK
Atlas → gateway: "pagou?"        gateway → Atlas: "o pedido 9042 foi pago"
        (a cada 5 min)                   (no instante em que acontece)
```

| | Polling | Webhook |
|---|---------|---------|
| Latência | até o intervalo | imediata |
| Chamadas desperdiçadas | 99% delas | zero |
| Quem precisa estar no ar | o gateway | **você** |
| Superfície de ataque | nenhuma | 🔴 **um endpoint público** |

> 🔴 **Este é o ponto que muda tudo.** Um webhook é uma URL sua, pública, que aceita `POST` de qualquer um na internet e **altera o estado do seu sistema**.
>
> Se você aceitar sem verificar, qualquer pessoa marca qualquer pedido como pago.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  O gateway de pagamento (fictício) — quem ENVIA o webhook
# ═══════════════════════════════════════════════════════════════
SEGREDO_WEBHOOK = "whsec_" + secrets.token_urlsafe(24)


def assinar(segredo: str, timestamp: str, corpo: bytes) -> str:
    """HMAC-SHA256 sobre `timestamp.corpo`.

    🔑 Por que o timestamp entra na assinatura? Para que ela não possa ser
       reaproveitada. Sem ele, quem interceptar uma notificação legítima
       pode reenviá-la mil vezes — e cada reenvio tem assinatura válida.

    💭 Este é o mesmo esquema que Stripe, GitHub e Shopify usam. O nome
       do cabeçalho muda; a ideia é idêntica.
    """
    mensagem = timestamp.encode() + b"." + corpo
    return hmac.new(segredo.encode(), mensagem, hashlib.sha256).hexdigest()


def montar_notificacao(evento: dict, segredo: str = SEGREDO_WEBHOOK,
                       timestamp: str | None = None) -> tuple[bytes, dict]:
    """Devolve (corpo_em_bytes, cabeçalhos) como o gateway enviaria."""
    corpo = json.dumps(evento, separators=(",", ":")).encode()
    ts = timestamp or str(int(time.time()))
    return corpo, {
        "Content-Type": "application/json",
        "X-Gateway-Timestamp": ts,
        "X-Gateway-Assinatura": assinar(segredo, ts, corpo),
    }


print(f"segredo compartilhado: {SEGREDO_WEBHOOK[:18]}…")
print("🔴 Ele vive no .env dos DOIS lados. Nunca no código.")

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  O Atlas — quem RECEBE
# ═══════════════════════════════════════════════════════════════
from fastapi import BackgroundTasks, FastAPI, Header, HTTPException, Request, status

atlas = FastAPI(title="Atlas — recepção de webhooks")

PEDIDOS = {9042: {"id": 9042, "status": "aguardando_pagamento", "valor": 2599.90},
           9043: {"id": 9043, "status": "aguardando_pagamento", "valor": 1199.00}}
EVENTOS_VISTOS: set[str] = set()
DIARIO: list[str] = []

TOLERANCIA_S = 300      # 5 minutos


def conferir_assinatura(corpo: bytes, timestamp: str | None, recebida: str | None):
    """As três verificações, nesta ordem."""
    if not timestamp or not recebida:
        raise HTTPException(status.HTTP_401_UNAUTHORIZED, "assinatura ausente")

    # 1️⃣ Frescor — impede reenvio de uma notificação antiga
    try:
        idade = abs(time.time() - int(timestamp))
    except ValueError:
        raise HTTPException(status.HTTP_401_UNAUTHORIZED, "timestamp inválido")
    if idade > TOLERANCIA_S:
        raise HTTPException(status.HTTP_401_UNAUTHORIZED,
                            f"notificação velha ({idade:.0f}s)")

    # 2️⃣ Autenticidade — só quem tem o segredo produz esta assinatura
    esperada = assinar(SEGREDO_WEBHOOK, timestamp, corpo)

    # 🔴 compare_digest, NÃO `==`.
    #    Um `==` comum para de comparar no primeiro byte diferente. A
    #    diferença de tempo é minúscula, mas mensurável — e permite
    #    descobrir a assinatura byte a byte. Chama-se ataque de tempo.
    if not hmac.compare_digest(esperada, recebida):
        raise HTTPException(status.HTTP_401_UNAUTHORIZED, "assinatura inválida")


def processar(evento: dict):
    """O trabalho de verdade — roda DEPOIS de responder."""
    time.sleep(0.05)
    pedido = PEDIDOS.get(evento["pedido_id"])
    if pedido and evento["tipo"] == "pagamento.aprovado":
        pedido["status"] = "pago"
        DIARIO.append(f"pedido {pedido['id']} → pago")


@atlas.post("/webhooks/gateway", status_code=status.HTTP_202_ACCEPTED)
async def receber(requisicao: Request, tarefas: BackgroundTasks,
                  x_gateway_timestamp: str | None = Header(default=None),
                  x_gateway_assinatura: str | None = Header(default=None)):
    """Recebe, valida, confirma rápido e processa depois.

    ⚠️ Lê o corpo CRU com `await requisicao.body()`, não como modelo
       Pydantic. A assinatura foi calculada sobre os bytes exatos —
       reserializar o JSON muda espaços e ordem de chaves, e a
       assinatura deixa de bater.
    """
    corpo = await requisicao.body()
    conferir_assinatura(corpo, x_gateway_timestamp, x_gateway_assinatura)

    evento = json.loads(corpo)

    # 3️⃣ Idempotência — o mesmo evento VAI chegar de novo
    id_evento = evento.get("id")
    if id_evento in EVENTOS_VISTOS:
        DIARIO.append(f"evento {id_evento} ignorado (repetido)")
        return {"recebido": True, "repetido": True}
    EVENTOS_VISTOS.add(id_evento)

    # 🔑 Responda AGORA, processe depois.
    tarefas.add_task(processar, evento)
    return {"recebido": True, "repetido": False}


atlas_cliente = TestClient(atlas)
print("✅ receptor pronto")

In [ ]:
# ── Notificação legítima ──
corpo, cabecalhos = montar_notificacao({
    "id": "evt_001", "tipo": "pagamento.aprovado",
    "pedido_id": 9042, "valor": 2599.90,
})
print("antes :", PEDIDOS[9042])
req(atlas_cliente, "POST", "/webhooks/gateway", content=corpo, headers=cabecalhos)
print("depois:", PEDIDOS[9042])

In [ ]:
# 🔴 Os três ataques que a validação impede
print("═══ 1. Sem assinatura (o atacante só descobriu a URL) ═══")
req(atlas_cliente, "POST", "/webhooks/gateway",
    json={"id": "evt_falso", "tipo": "pagamento.aprovado", "pedido_id": 9043})

print("═══ 2. Assinatura forjada (segredo errado) ═══")
corpo_f, cab_f = montar_notificacao(
    {"id": "evt_falso2", "tipo": "pagamento.aprovado", "pedido_id": 9043},
    segredo="chute-do-atacante")
req(atlas_cliente, "POST", "/webhooks/gateway", content=corpo_f, headers=cab_f)

print("═══ 3. Reenvio de notificação antiga (replay) ═══")
antigo = str(int(time.time()) - 3600)
corpo_v, cab_v = montar_notificacao(
    {"id": "evt_velho", "tipo": "pagamento.aprovado", "pedido_id": 9043},
    timestamp=antigo)
req(atlas_cliente, "POST", "/webhooks/gateway", content=corpo_v, headers=cab_v)

print(f"pedido 9043 continua: {PEDIDOS[9043]['status']}")

In [ ]:
# ⚠️ E o evento legítimo que chega DUAS vezes?
print("O gateway não recebeu nossa confirmação e reenviou:\n")
for tentativa in (1, 2, 3):
    r = atlas_cliente.post("/webhooks/gateway", content=corpo, headers=cabecalhos)
    print(f"   envio {tentativa}: {r.status_code}  {r.json()}")

time.sleep(0.3)
print("\nDiário do processamento:")
for linha in DIARIO:
    print(f"   · {linha}")

> 🔴 **Reenvio não é hipótese — é o funcionamento normal.**
>
> O gateway espera a sua confirmação. Se ela não chegar em alguns segundos (a sua rede engasgou, o seu servidor reiniciou, a resposta se perdeu), ele **reenvia**. Stripe reenvia por até 3 dias.
>
> Sem a checagem de `id`, cada reenvio marcaria o pedido como pago de novo — e se houvesse um efeito colateral (mandar e-mail, creditar bônus, baixar estoque), ele aconteceria várias vezes.
>
> 🧭 **Toda recepção de webhook precisa de:**
>
> | # | Verificação | Contra o quê |
> |---|-------------|--------------|
> | 1 | Assinatura HMAC com `compare_digest` | Falsificação |
> | 2 | Timestamp dentro da janela | Reenvio malicioso |
> | 3 | `id` do evento já visto | Reenvio legítimo |
> | 4 | Responder rápido (`202`) | Timeout do remetente |
>
> ⚠️ **Em produção, `EVENTOS_VISTOS` não pode ser um `set` em memória** — ele se perde no restart e não é compartilhado entre instâncias. Use uma tabela com `UNIQUE` no id, ou uma chave no Redis com TTL.

In [ ]:
# 💡 Por que responder rápido importa
lento_app = FastAPI()


@lento_app.post("/sincrono")
async def sincrono(requisicao: Request):
    """🔴 Processa ANTES de responder."""
    await requisicao.body()
    time.sleep(0.5)              # imagine gerar PDF, mandar e-mail...
    return {"ok": True}


@lento_app.post("/com-background")
async def com_background(requisicao: Request, tarefas: BackgroundTasks):
    """✅ Confirma e processa depois."""
    await requisicao.body()
    tarefas.add_task(time.sleep, 0.5)
    return {"ok": True}


# ⚠️ ARMADILHA DE TESTE: o TestClient ESPERA as tarefas de segundo plano
#    terminarem antes de devolver a resposta. Ele é síncrono por dentro.
#    Medido por ele, os dois números saem iguais — e você conclui,
#    erradamente, que o BackgroundTasks não serve para nada.
c = TestClient(lento_app)
print("Medido pelo TestClient:")
for rota in ("/sincrono", "/com-background"):
    inicio = time.perf_counter()
    c.post(rota, json={})
    print(f"   {rota:<18} {(time.perf_counter() - inicio) * 1000:6.0f} ms")
print("   ↑ iguais! Porque o TestClient aguarda as tarefas.\n")

# ✅ Contra um servidor DE VERDADE, a diferença aparece
lento_servico = Servico(lento_app, "app lenta").iniciar()
print("\nMedido contra um servidor real:")
for rota in ("/sincrono", "/com-background"):
    inicio = time.perf_counter()
    httpx.post(f"{lento_servico.url}{rota}", json={}, timeout=10)
    print(f"   {rota:<18} {(time.perf_counter() - inicio) * 1000:6.0f} ms")
lento_servico.parar()

print("\n💡 O gateway costuma desistir entre 5 e 30 segundos.")
print("   E cada desistência dele é um reenvio para você.")

> ⚠️ **Guarde esta armadilha — ela vale para a aula 07_03.**
>
> O `TestClient` executa as tarefas de segundo plano **antes** de devolver a resposta. Isso é ótimo para testar (você pode verificar o efeito logo depois do `post`), mas significa que **ele não mede latência de verdade** e não reproduz o comportamento do servidor real.
>
> Se você medir desempenho com `TestClient`, vai medir a coisa errada.

In [ ]:
# 🔴 E o lado perigoso da mesma moeda: a tarefa que se perde
frageis = FastAPI()
FEITAS: list[str] = []


@frageis.post("/importante")
async def importante(tarefas: BackgroundTasks):
    tarefas.add_task(FEITAS.append, "efeito colateral que ninguém viu falhar")
    return {"aceito": True}


def tarefa_que_falha():
    raise RuntimeError("o SMTP recusou a conexão")


@frageis.post("/que-falha")
async def que_falha(tarefas: BackgroundTasks):
    tarefas.add_task(tarefa_que_falha)
    return {"aceito": True}


c = TestClient(frageis, raise_server_exceptions=False)
r = c.post("/importante")
print(f"/importante  → {r.status_code} {r.json()}   feitas: {FEITAS}")

r = c.post("/que-falha", )
print(f"/que-falha   → {r.status_code} {r.json()}")
print("\n🔴 A tarefa estourou DEPOIS da resposta.")
print("   O cliente recebeu 200. Ninguém foi avisado. Não há retentativa.")
print("   Em produção, isso é um e-mail de confirmação que nunca saiu —")
print("   e você só descobre quando o cliente liga.")

## 2. Upload de arquivos

O setor de compras quer enviar a planilha de reposição pelo sistema.

In [ ]:
from fastapi import File, UploadFile

# ═══ Assinaturas de arquivo (magic bytes) ═══
# 🔴 A extensão é só um pedaço do NOME — o usuário escolhe o que quiser.
#    Os primeiros bytes são o que o arquivo REALMENTE é.
ASSINATURAS = {
    b"PK\x03\x04": "zip/xlsx/docx",
    b"\x89PNG\r\n\x1a\n": "png",
    b"%PDF-": "pdf",
    b"\xff\xd8\xff": "jpeg",
    # ── executáveis: aqui só para serem RECONHECIDOS e recusados ──
    b"\x7fELF": "executavel-linux",
    b"MZ": "executavel-windows",
    b"\xcf\xfa\xed\xfe": "executavel-macos",
    b"#!": "script-shell",
}

TIPOS_ACEITOS = {"zip/xlsx/docx", "pdf", "csv/texto"}
TAMANHO_MAXIMO = 2 * 1024 * 1024        # 2 MB


def identificar(cabeca: bytes) -> str:
    """Descobre o que o arquivo é, pelos primeiros bytes.

    ⚠️ A ORDEM importa: as assinaturas conhecidas vêm primeiro, e só
       depois o palpite de "é texto".

    🔴 E o palpite de texto precisa ser ESTRITO. Um `.decode("utf-8")`
       que não estoura NÃO significa "é um CSV" — um binário ELF começa
       com bytes que são UTF-8 perfeitamente válidos. Foi exatamente
       assim que a primeira versão desta função deixou passar um
       executável.
    """
    for magico, tipo in ASSINATURAS.items():
        if cabeca.startswith(magico):
            return tipo

    try:
        texto = cabeca.decode("utf-8")
    except UnicodeDecodeError:
        return "binario-desconhecido"

    # 🔒 Texto de verdade não tem byte nulo nem caractere de controle
    #    (fora de tab, LF e CR).
    if any(c == "\x00" or (ord(c) < 32 and c not in "\t\n\r") for c in texto):
        return "binario-desconhecido"
    return "csv/texto"


def nome_seguro(nome: str) -> str:
    """🔴 Contra path traversal.

    `../../etc/passwd` ou `..\\..\\Windows\\System32\\config` no nome do
    arquivo faz o seu `open(pasta / nome, "wb")` escrever FORA da pasta.

    `Path(nome).name` descarta qualquer diretório. Depois filtramos o
    que sobrou para caracteres inofensivos.
    """
    base = Path(nome).name
    limpo = "".join(c for c in base if c.isalnum() or c in "._- ")
    return limpo.strip(". ") or "sem_nome"


for tentativa in ["planilha.xlsx", "../../etc/passwd", "..\\..\\senha.txt",
                  "rela;rm -rf /.csv", "....//....//x.csv", ".", ""]:
    print(f"   {tentativa!r:<28} → {nome_seguro(tentativa)!r}")

In [ ]:
uploads = FastAPI()
DESTINO = TRABALHO / "recebidos"


@uploads.post("/arquivos", status_code=201)
async def enviar(arquivo: UploadFile = File(...)):
    """Upload com as três defesas.

    ⚠️ `UploadFile` (e não `bytes`) porque o FastAPI o mantém em disco
       acima de ~1 MB, em vez de carregar tudo na memória. Um parâmetro
       `bytes` com um arquivo de 2 GB derruba o processo.
    """
    # 1️⃣ Tamanho — lendo em pedaços, sem carregar tudo
    tamanho = 0
    pedacos = []
    while pedaco := await arquivo.read(64 * 1024):
        tamanho += len(pedaco)
        if tamanho > TAMANHO_MAXIMO:
            raise HTTPException(413, f"máximo {TAMANHO_MAXIMO // 1024} KB")
        pedacos.append(pedaco)
    conteudo = b"".join(pedacos)

    if not conteudo:
        raise HTTPException(422, "arquivo vazio")

    # 2️⃣ Conteúdo real, não a extensão nem o Content-Type
    tipo = identificar(conteudo[:16])
    if tipo not in TIPOS_ACEITOS:
        raise HTTPException(
            415, f"conteúdo é {tipo}; aceitamos {sorted(TIPOS_ACEITOS)}")

    # 3️⃣ Nome sanitizado
    destino = DESTINO / nome_seguro(arquivo.filename or "")
    destino.write_bytes(conteudo)

    return {"nome_original": arquivo.filename, "nome_gravado": destino.name,
            "bytes": tamanho, "tipo_declarado": arquivo.content_type,
            "tipo_real": tipo}


cliente_up = TestClient(uploads)

csv_bom = b"sku,quantidade\nNB-DELL-15,10\nMO-LG-24UW,4\n"
req(cliente_up, "POST", "/arquivos",
    files={"arquivo": ("reposicao.csv", csv_bom, "text/csv")})

In [ ]:
# 🔴 Os ataques
print("═══ 1. Executável disfarçado de CSV ═══")
req(cliente_up, "POST", "/arquivos",
    files={"arquivo": ("inofensivo.csv", b"\x7fELF\x02\x01\x01\x00" + b"\x00" * 40,
                       "text/csv")})

print("═══ 2. Path traversal no nome ═══")
r = req(cliente_up, "POST", "/arquivos",
        files={"arquivo": ("../../../etc/passwd", csv_bom, "text/csv")})
print("   ⚠️  Repare: 201, mas o `nome_gravado` já vem SANITIZADO.")
print("   O upload foi aceito — o que NÃO aconteceu foi escrever fora da pasta.\n")

print("═══ 3. Arquivo grande demais ═══")
req(cliente_up, "POST", "/arquivos",
    files={"arquivo": ("gigante.csv", b"a,b\n" * 700_000, "text/csv")})

print("Onde os arquivos realmente foram parar:")
for p in sorted(DESTINO.rglob("*")):
    dentro = DESTINO.resolve() in p.resolve().parents
    print(f"   {'✅' if dentro else '🔴'} {p.resolve()}  ({p.stat().st_size} bytes)")

print(f"\n✅ /etc/passwd continua intacto: nada foi escrito fora de {DESTINO.name}/")

> 🔴 **Três vulnerabilidades num único endpoint de upload:**
>
> | Ataque | Defesa | Por que a defesa óbvia falha |
> |--------|--------|------------------------------|
> | Executável disfarçado | **Magic bytes** | Extensão e `Content-Type` vêm do cliente |
> | Path traversal | `Path(nome).name` + filtro | Concatenar caminho é o bug |
> | Esgotar disco/memória | Ler em pedaços com teto | `await arquivo.read()` inteiro é o bug |
>
> ⚠️ **O `Content-Type` é uma sugestão do cliente, não um fato.** O navegador o preenche a partir da extensão; um script preenche com o que quiser.
>
> 🔴 **Ainda falta uma quarta defesa:** não sirva de volta o que foi enviado a partir do mesmo domínio da aplicação. Um HTML enviado e servido em `seusite.com/uploads/x.html` roda JavaScript **no seu domínio**, com acesso aos cookies. Sirva de outro domínio, ou force `Content-Disposition: attachment` com `X-Content-Type-Options: nosniff`.

## 3. Download e streaming

In [ ]:
from fastapi.responses import FileResponse, StreamingResponse

downloads = FastAPI()
ARQUIVOS = {"reposicao.csv": DESTINO / "reposicao.csv"}


@downloads.get("/arquivos/{nome}")
def baixar(nome: str):
    """🔴 A MESMA armadilha de path traversal, na direção contrária."""
    caminho = ARQUIVOS.get(nome_seguro(nome))
    if caminho is None or not caminho.exists():
        raise HTTPException(404, "arquivo não encontrado")
    return FileResponse(caminho, media_type="text/csv", filename=nome)


@downloads.get("/relatorio.csv")
def relatorio_grande():
    """Gera e envia em fluxo, sem montar tudo na memória.

    🎯 O gerador é o que permite exportar 5 milhões de linhas com uso de
       memória constante. O primeiro byte chega ao cliente antes de a
       última linha existir.
    """
    def linhas():
        yield "sku,categoria,quantidade,valor\n"
        for i in range(1, 50_001):
            yield f"SKU-{i:05d},cat{i % 7},{i % 30},{i * 1.5:.2f}\n"

    return StreamingResponse(
        linhas(),
        media_type="text/csv",
        # 🔑 attachment = o navegador baixa em vez de exibir
        headers={"Content-Disposition": 'attachment; filename="relatorio.csv"'},
    )


c = TestClient(downloads)

r = c.get("/arquivos/reposicao.csv")
print(f"✅ download simples: {r.status_code}, {len(r.content)} bytes")
print(f"   {r.headers.get('content-disposition')}")

r = c.get("/arquivos/..%2F..%2Fetc%2Fpasswd")
print(f"\n🔴 tentativa de traversal: {r.status_code}")

r = c.get("/relatorio.csv")
linhas = r.text.count("\n")
print(f"\n✅ streaming: {r.status_code}, {linhas:,} linhas, "
      f"{len(r.content) / 1024:.0f} KB")
print(f"   primeira linha de dados: {r.text.splitlines()[1]}")

> 💡 **`FileResponse` vs `StreamingResponse`:**
>
> | | Use quando |
> |---|-----------|
> | `FileResponse` | O arquivo **já existe** em disco. Ele cuida de `Content-Length`, `Last-Modified` e requisições parciais. |
> | `StreamingResponse` | Você **gera** o conteúdo. Memória constante, mas sem `Content-Length` — o navegador não mostra a barra de progresso. |
>
> ⚠️ **Não use `StreamingResponse` com um gerador que abre uma sessão de banco.** A sessão precisa continuar viva enquanto o gerador produz, e o `get_sessao` do M06 já a fechou quando a rota retornou. É uma das causas mais confusas de `DetachedInstanceError`.

## 4. `BackgroundTasks` — e seus limites

In [ ]:
tarefas_app = FastAPI()
REGISTRO: list[str] = []


def enviar_email(destino: str, assunto: str):
    time.sleep(0.1)
    REGISTRO.append(f"e-mail → {destino}: {assunto}")


def atualizar_estoque(sku: str, delta: int):
    REGISTRO.append(f"estoque {sku} {delta:+d}")


@tarefas_app.post("/pedidos/{pedido_id}/confirmar")
def confirmar(pedido_id: int, tarefas: BackgroundTasks):
    """Responde imediatamente; os efeitos acontecem em seguida.

    ⚠️ As tarefas rodam DEPOIS de a resposta ser enviada, mas NO MESMO
       processo. Se o processo morrer, elas se perdem — sem aviso.
    """
    tarefas.add_task(atualizar_estoque, "NB-DELL-15", -1)
    tarefas.add_task(enviar_email, "cliente@aurora.com.br", f"Pedido {pedido_id}")
    tarefas.add_task(enviar_email, "compras@aurora.com.br", "Estoque baixo")
    return {"pedido": pedido_id, "status": "confirmado"}


c = TestClient(tarefas_app)
inicio = time.perf_counter()
r = c.post("/pedidos/9042/confirmar")
ms = (time.perf_counter() - inicio) * 1000

print(f"resposta em {ms:.0f} ms: {r.json()}")
time.sleep(0.4)
print("\nO que rodou depois:")
for linha in REGISTRO:
    print(f"   · {linha}")

> 🔴 **`BackgroundTasks` NÃO é uma fila.** Ele é um `await` adiado, no mesmo processo.
>
> | | `BackgroundTasks` | Fila (Celery, RQ, Dramatiq) |
> |---|-------------------|------------------------------|
> | Sobrevive ao restart | ❌ **perde tudo** | ✅ persistida |
> | Retentativa automática | ❌ | ✅ |
> | Visibilidade do que falhou | ❌ | ✅ |
> | Distribui entre máquinas | ❌ | ✅ |
> | Tarefa de 10 minutos | ❌ prende o worker | ✅ |
> | Custo de operação | zero | 🔶 mais um serviço |
>
> 🧭 **A régua:**
>
> - **`BackgroundTasks`** para o que é rápido (< 1 s) e cuja perda é tolerável: log, métrica, invalidar cache, notificação de conveniência.
> - **Fila de verdade** para o que é lento ou cuja perda dói: e-mail de confirmação de compra, geração de nota fiscal, processamento de pagamento.
>
> 💭 **A pergunta que decide:** *"se esta tarefa nunca rodar, e ninguém for avisado, qual o prejuízo?"* Se a resposta não for "nenhum", você precisa de uma fila.
>
> Filas com Redis e Celery são assunto do Módulo 10.

## 5. Cache

A tela de estoque faz a mesma pergunta 900 vezes por minuto.

In [ ]:
cache = abrir_cache()
cache.flushall()

consultas_ao_banco = {"n": 0}


def consultar_banco(sku: str) -> dict:
    """Simula uma consulta cara."""
    consultas_ao_banco["n"] += 1
    time.sleep(0.05)
    return {"sku": sku, "estoque": 14, "reservado": 2,
            "consultado_em": datetime.now(timezone.utc).isoformat(timespec="seconds")}


def com_cache(sku: str, ttl: int = 30) -> tuple[dict, bool]:
    """Cache-aside: o padrão mais comum, e o mais simples de acertar.

        1. olha no cache
        2. se não achou, busca na fonte
        3. grava no cache com TTL
    """
    chave = f"atlas:estoque:v1:{sku}"        # 🔑 veja a nota sobre o `v1`
    guardado = cache.get(chave)
    if guardado is not None:
        return json.loads(guardado), True
    dado = consultar_banco(sku)
    cache.setex(chave, ttl, json.dumps(dado))
    return dado, False


print("30 consultas ao mesmo SKU:\n")
inicio = time.perf_counter()
acertos = 0
for _ in range(30):
    _, veio_do_cache = com_cache("NB-DELL-15")
    acertos += veio_do_cache
ms = (time.perf_counter() - inicio) * 1000

print(f"   consultas ao banco : {consultas_ao_banco['n']}")
print(f"   acertos de cache   : {acertos}/30")
print(f"   tempo total        : {ms:.0f} ms")
print(f"   sem cache seriam   : {30 * 50} ms")

In [ ]:
# 🔑 O desenho da CHAVE é metade do trabalho
exemplos = [
    ("atlas:estoque:v1:NB-DELL-15",             "✅ namespace:recurso:versão:id"),
    ("atlas:relatorio:v1:faturamento:2026-08",  "✅ período no lugar do 'hoje'"),
    ("atlas:produtos:v1:cat=Notebooks&ord=preco", "✅ filtros na chave"),
    ("NB-DELL-15",                              "🔴 colide com qualquer outro sistema"),
    ("estoque",                                 "🔴 uma chave para todos os SKUs"),
    ("atlas:relatorio:hoje",                    "🔴 'hoje' muda de significado"),
]
for chave, comentario in exemplos:
    print(f"   {chave:<44} {comentario}")

print("\n💡 O `v1` é o truque mais subestimado:")
print("   mudou o formato do que você guarda? incremente para v2.")
print("   As chaves v1 expiram sozinhas e você nunca lê um formato antigo")
print("   como se fosse o novo — que é um bug particularmente confuso.")

In [ ]:
# ⚠️ Invalidação: o problema difícil
def gravar_estoque(sku: str, novo: int):
    """🔑 Escreveu na fonte? Invalide o cache NA MESMA operação."""
    # ... UPDATE produtos SET estoque = ? WHERE sku = ?
    cache.delete(f"atlas:estoque:v1:{sku}")
    print(f"   estoque de {sku} atualizado e cache invalidado")


dado, _ = com_cache("MO-LG-24UW")
print(f"1. leitura   → estoque {dado['estoque']}, do banco")
dado, veio = com_cache("MO-LG-24UW")
print(f"2. leitura   → cache? {veio}")
gravar_estoque("MO-LG-24UW", 99)
dado, veio = com_cache("MO-LG-24UW")
print(f"3. leitura   → cache? {veio}  (foi buscar de novo ✅)")

print(f"\ntotal de consultas ao banco: {consultas_ao_banco['n']}")

> 🎯 **"Só existem duas coisas difíceis em computação: invalidação de cache e dar nomes às coisas."** — Phil Karlton
>
> A piada é famosa porque é verdade. **Cache é uma cópia**, e toda cópia pode divergir do original. A pergunta não é *"como evito divergir?"* — é *"por quanto tempo posso divergir sem causar dano?"*
>
> | Dado | Divergir por | Por quê |
> |------|--------------|---------|
> | Nome/descrição do produto | horas | muda raramente |
> | Preço | minutos | promoção precisa entrar rápido |
> | **Estoque** | 🔴 segundos ou nada | vender o que não tem custa dinheiro |
> | Saldo financeiro | 🔴 nunca cacheie | |
>
> 🧭 **Duas estratégias, e uma regra:**
>
> - **TTL** — expira sozinho. Simples, sempre funciona, aceita divergência.
> - **Invalidação explícita** — apaga ao escrever. Preciso, mas é fácil esquecer um caminho de escrita.
>
> **Use os dois.** A invalidação cuida do caso normal; o TTL é a rede de segurança para o caminho que você esqueceu.

In [ ]:
# 🔴 O ESTOURO DE CACHE (cache stampede)
cache.flushall()
consultas_ao_banco["n"] = 0

import threading as _th

def sem_protecao(sku="AR-KING-1TB"):
    com_cache(sku, ttl=30)

# 20 requisições simultâneas, com o cache vazio
threads = [_th.Thread(target=sem_protecao) for _ in range(20)]
for t in threads: t.start()
for t in threads: t.join()

print(f"🔴 20 requisições simultâneas → {consultas_ao_banco['n']} consultas ao banco")
print("   Todas erraram o cache ao mesmo tempo e foram TODAS ao banco.\n")

# ✅ Com trava: só uma vai à fonte
cache.flushall()
consultas_ao_banco["n"] = 0
_travas: dict[str, _th.Lock] = {}
_trava_global = _th.Lock()


def com_trava(sku: str, ttl: int = 30):
    chave = f"atlas:estoque:v1:{sku}"
    guardado = cache.get(chave)
    if guardado is not None:
        return json.loads(guardado)

    with _trava_global:
        trava = _travas.setdefault(sku, _th.Lock())
    with trava:
        # 🔑 confere DE NOVO: outra thread pode ter preenchido enquanto
        #    esperávamos a trava. Sem esta segunda checagem, a trava
        #    apenas enfileira as consultas em vez de evitá-las.
        guardado = cache.get(chave)
        if guardado is not None:
            return json.loads(guardado)
        dado = consultar_banco(sku)
        cache.setex(chave, ttl, json.dumps(dado))
        return dado


threads = [_th.Thread(target=lambda: com_trava("AR-KING-1TB")) for _ in range(20)]
for t in threads: t.start()
for t in threads: t.join()

print(f"✅ 20 requisições simultâneas → {consultas_ao_banco['n']} consulta ao banco")

> 🔴 **O estouro acontece exatamente quando você menos pode pagar por ele.**
>
> Uma chave popular expira. Nesse instante, todas as requisições em voo erram o cache e vão juntas ao banco. Com tráfego alto, isso é uma **avalanche** — e o banco cai justamente na hora de pico.
>
> **Três defesas, combináveis:**
>
> 1. **Trava** (o exemplo acima) — só um recalcula, os outros esperam
> 2. **TTL com jitter** — `ttl + random(0, 60)` faz as chaves expirarem espalhadas
> 3. **Recomputação antecipada** — renova antes de expirar, com o valor velho ainda servindo
>
> ⚠️ **Numa API com várias instâncias, a trava precisa ser distribuída** — um `threading.Lock` só protege dentro do processo. O Redis resolve isso com `SET chave valor NX EX 10` (grava só se não existir).

In [ ]:
# ⚠️ O que o fakeredis NÃO reproduz fielmente
print("Funciona igual:")
for op in ["get/set/setex", "delete", "incr/decr", "expire/ttl",
           "hash e list", "pipeline", "scan"]:
    print(f"   ✅ {op}")

print("\nDiferente ou ausente:")
for op, motivo in [
    ("cluster e replicação", "é um único processo"),
    ("eviction sob pressão", "não simula limite de memória (maxmemory)"),
    ("latência de rede",     "🔴 zero — mede errado o custo real"),
    ("persistência",         "some ao reiniciar o kernel"),
    ("scripts Lua",          "suporte parcial"),
]:
    print(f"   ⚠️  {op:<22} {motivo}")

print("\n💭 Como no M05 com o mongomock: use o simulacro para APRENDER e")
print("   para TESTAR. Antes de produção, rode contra o Redis de verdade.")

## 6. WebSockets

HTTP é pergunta-e-resposta. Para o servidor **empurrar** informação, existe o WebSocket.

In [ ]:
from fastapi import WebSocket, WebSocketDisconnect

tempo_real = FastAPI()


class Conexoes:
    """Gerencia quem está conectado.

    ⚠️ Em memória: com várias instâncias da API, cada uma conhece só os
       seus. Broadcast de verdade exige um canal compartilhado — o
       Pub/Sub do Redis é a solução usual.
    """

    def __init__(self):
        self.ativas: list[WebSocket] = []

    async def conectar(self, sock: WebSocket):
        await sock.accept()
        self.ativas.append(sock)

    def desconectar(self, sock: WebSocket):
        if sock in self.ativas:
            self.ativas.remove(sock)

    async def transmitir(self, mensagem: dict):
        # ⚠️ Itere sobre uma CÓPIA: um envio pode falhar e remover o
        #    socket da lista durante a própria iteração.
        for sock in list(self.ativas):
            try:
                await sock.send_json(mensagem)
            except Exception:
                self.desconectar(sock)


conexoes = Conexoes()
ESTOQUE = {"NB-DELL-15": 14, "MO-LG-24UW": 31}


@tempo_real.websocket("/ws/estoque")
async def painel(sock: WebSocket):
    await conexoes.conectar(sock)
    try:
        await sock.send_json({"tipo": "inicial", "estoque": dict(ESTOQUE)})
        while True:
            comando = await sock.receive_json()
            if comando.get("acao") == "assinar":
                await sock.send_json({"tipo": "assinado",
                                      "skus": comando.get("skus", [])})
            elif comando.get("acao") == "sair":
                break
    except WebSocketDisconnect:
        pass
    finally:
        conexoes.desconectar(sock)


@tempo_real.post("/estoque/{sku}")
async def mudar(sku: str, delta: int):
    """Uma rota HTTP normal que EMPURRA a mudança para os painéis."""
    ESTOQUE[sku] = ESTOQUE.get(sku, 0) + delta
    await conexoes.transmitir({"tipo": "mudou", "sku": sku,
                               "estoque": ESTOQUE[sku]})
    return {"sku": sku, "estoque": ESTOQUE[sku]}


c = TestClient(tempo_real)

with c.websocket_connect("/ws/estoque") as painel_a:
    print("📡 painel A conectado")
    print("   recebeu:", painel_a.receive_json())

    painel_a.send_json({"acao": "assinar", "skus": ["NB-DELL-15"]})
    print("   recebeu:", painel_a.receive_json())

    c.post("/estoque/NB-DELL-15?delta=-3")
    print("   🔔 empurrado:", painel_a.receive_json())

    c.post("/estoque/MO-LG-24UW?delta=5")
    print("   🔔 empurrado:", painel_a.receive_json())

    painel_a.send_json({"acao": "sair"})

print(f"\nconexões ativas depois: {len(conexoes.ativas)}")

> 🎯 **Quando WebSocket vale a pena?**
>
> | Situação | Escolha |
> |----------|---------|
> | Atualiza a cada minuto | HTTP normal, com cache |
> | Atualiza a cada 5–30 s | *Polling* — mais simples, funciona em qualquer lugar |
> | Vários eventos por segundo | **WebSocket** |
> | Só o servidor fala (notificações) | **SSE** (`text/event-stream`) — metade da complexidade |
> | Chat, colaboração, jogo | **WebSocket** |
>
> ⚠️ **O custo escondido:** WebSocket é uma conexão **permanente**. 10.000 usuários = 10.000 conexões abertas, cada uma ocupando memória e um descritor de arquivo. Proxies reversos precisam de configuração especial. Balanceadores precisam de sessão fixa. Reconexão é problema seu.
>
> 💭 **A pergunta honesta:** *"polling a cada 10 segundos resolve?"* Quase sempre resolve — e custa uma linha de JavaScript.
>
> 🔴 **Autenticação em WebSocket é diferente.** O navegador não permite cabeçalhos customizados no *handshake*, então `Authorization: Bearer` não funciona direto. As saídas são: token na query string (⚠️ vaza no log do servidor), cookie, ou uma primeira mensagem de autenticação após conectar.

## 🔧 Prática guiada — recepção de pagamentos ponta a ponta

Vamos juntar webhook, idempotência, cache e background numa única aplicação — e mandar o gateway **de verdade** chamar o Atlas por HTTP.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Atlas — versão completa
# ═══════════════════════════════════════════════════════════════
cache.flushall()
final = FastAPI(title="Atlas — integrações")

BANCO_PEDIDOS = {n: {"id": n, "status": "aguardando_pagamento",
                     "valor": round(500 + n * 1.7, 2)} for n in range(9040, 9050)}
AUDITORIA: list[dict] = []


def registrar(evento: str, **campos):
    AUDITORIA.append({"em": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                      "evento": evento, **campos})


def ja_processado(id_evento: str, ttl: int = 86400) -> bool:
    """Idempotência com Redis.

    🔑 `SET chave valor NX EX ttl` grava SÓ SE não existir, de forma
       atômica. Devolve None quando a chave já estava lá — e é isso que
       torna esta checagem segura mesmo com várias instâncias da API
       recebendo o mesmo reenvio ao mesmo tempo.
    """
    return cache.set(f"atlas:webhook:visto:{id_evento}", "1",
                     nx=True, ex=ttl) is None


def liquidar(evento: dict):
    pedido = BANCO_PEDIDOS.get(evento["pedido_id"])
    if pedido is None:
        registrar("pedido_inexistente", pedido_id=evento["pedido_id"])
        return
    if evento["tipo"] == "pagamento.aprovado":
        pedido["status"] = "pago"
        cache.delete(f"atlas:pedido:v1:{pedido['id']}")     # invalida
        registrar("liquidado", pedido_id=pedido["id"], valor=evento["valor"])
    elif evento["tipo"] == "pagamento.recusado":
        pedido["status"] = "recusado"
        cache.delete(f"atlas:pedido:v1:{pedido['id']}")
        registrar("recusado", pedido_id=pedido["id"])


@final.post("/webhooks/gateway", status_code=202)
async def webhook(requisicao: Request, tarefas: BackgroundTasks,
                  x_gateway_timestamp: str | None = Header(default=None),
                  x_gateway_assinatura: str | None = Header(default=None)):
    corpo = await requisicao.body()
    conferir_assinatura(corpo, x_gateway_timestamp, x_gateway_assinatura)
    evento = json.loads(corpo)

    if ja_processado(evento["id"]):
        registrar("repetido", id_evento=evento["id"])
        return {"recebido": True, "repetido": True}

    tarefas.add_task(liquidar, evento)
    return {"recebido": True, "repetido": False}


@final.get("/pedidos/{pedido_id}")
def ver_pedido(pedido_id: int):
    """Leitura cacheada por 30 s."""
    chave = f"atlas:pedido:v1:{pedido_id}"
    if (guardado := cache.get(chave)) is not None:
        return {**json.loads(guardado), "_cache": True}
    pedido = BANCO_PEDIDOS.get(pedido_id)
    if pedido is None:
        raise HTTPException(404, "pedido não encontrado")
    time.sleep(0.03)
    cache.setex(chave, 30, json.dumps(pedido))
    return {**pedido, "_cache": False}


@final.get("/auditoria")
def ver_auditoria():
    return {"total": len(AUDITORIA), "eventos": AUDITORIA[-10:]}


atlas_servico = Servico(final, "Atlas").iniciar()
ATLAS_URL = atlas_servico.url

In [ ]:
# ═══ O gateway chama o Atlas por HTTP DE VERDADE ═══
def gateway_notifica(evento: dict, tentativas: int = 3) -> httpx.Response:
    """Como um gateway real faria: envia e repete se não confirmarmos."""
    corpo, cabecalhos = montar_notificacao(evento)
    for tentativa in range(1, tentativas + 1):
        try:
            r = httpx.post(f"{ATLAS_URL}/webhooks/gateway",
                           content=corpo, headers=cabecalhos, timeout=5.0)
            if r.status_code < 300:
                return r
            print(f"      gateway: tentativa {tentativa} → {r.status_code}, repetindo")
        except httpx.RequestError as erro:
            print(f"      gateway: tentativa {tentativa} → {type(erro).__name__}")
        time.sleep(0.2)
    return r


print("── pagamento aprovado ──")
antes = httpx.get(f"{ATLAS_URL}/pedidos/9042").json()
print(f"   antes: {antes['status']}  (cache={antes['_cache']})")

r = gateway_notifica({"id": "evt_100", "tipo": "pagamento.aprovado",
                      "pedido_id": 9042, "valor": 2599.90})
print(f"   webhook: {r.status_code} {r.json()}")
time.sleep(0.3)

depois = httpx.get(f"{ATLAS_URL}/pedidos/9042").json()
print(f"   depois: {depois['status']}  (cache={depois['_cache']})")

In [ ]:
print("── o gateway reenvia o MESMO evento 4 vezes ──")
for i in range(1, 5):
    r = gateway_notifica({"id": "evt_100", "tipo": "pagamento.aprovado",
                          "pedido_id": 9042, "valor": 2599.90})
    print(f"   envio {i}: {r.json()}")

print("\n── pagamento recusado, outro pedido ──")
gateway_notifica({"id": "evt_101", "tipo": "pagamento.recusado",
                  "pedido_id": 9043, "valor": 1199.00})
time.sleep(0.3)
print(f"   pedido 9043: {httpx.get(f'{ATLAS_URL}/pedidos/9043').json()['status']}")

print("\n── auditoria ──")
for e in httpx.get(f"{ATLAS_URL}/auditoria").json()["eventos"]:
    extras = {k: v for k, v in e.items() if k not in ("em", "evento")}
    print(f"   {e['em']}  {e['evento']:<20} {extras}")

In [ ]:
# Cache em ação
print("10 leituras do pedido 9042:\n")
inicio = time.perf_counter()
do_cache = 0
for _ in range(10):
    do_cache += httpx.get(f"{ATLAS_URL}/pedidos/9042").json()["_cache"]
ms = (time.perf_counter() - inicio) * 1000
print(f"   {do_cache}/10 vieram do cache, em {ms:.0f} ms")

# Um webhook novo invalida
gateway_notifica({"id": "evt_102", "tipo": "pagamento.aprovado",
                  "pedido_id": 9042, "valor": 2599.90})
time.sleep(0.3)
print(f"\n   após o webhook: _cache = "
      f"{httpx.get(f'{ATLAS_URL}/pedidos/9042').json()['_cache']}  ← invalidado ✅")

In [ ]:
# Encerramento
atlas_servico.parar()
cache.flushall()
print("✅ tudo encerrado")

## 📝 Exercícios

**E1.** 🔴 Implemente a validação de webhook completa (assinatura, timestamp, idempotência) e escreva um teste para cada um dos três ataques.

**E2.** Mostre, com `time.perf_counter()`, a diferença entre `==` e `hmac.compare_digest` comparando assinaturas que diferem no primeiro byte vs no último. (Dica: você vai precisar de muitas repetições para o ruído sumir.)

**E3.** Troque o `set` de eventos vistos por uma tabela SQL com `UNIQUE`. Explique por que o `set` não serve em produção.

**E4.** Implemente o outro lado: um **emissor** de webhook com retry, backoff e uma fila de entregas que falharam.

**E5.** Adicione ao receptor uma resposta `202` com `Location` apontando para o status do processamento.

**E6.** 🔴 Escreva um upload que valide magic bytes, tamanho em streaming e nome. Tente burlar cada defesa e mostre a rejeição.

**E7.** Implemente upload em pedaços (*chunked*): o cliente envia um arquivo de 50 MB em fatias de 1 MB, retomável.

**E8.** Crie um `StreamingResponse` que gere 1 milhão de linhas de CSV. Meça a memória do processo durante a geração e prove que ela é constante.

**E9.** 🔴 Demonstre o path traversal: escreva a versão INSEGURA, mostre-a escrevendo fora da pasta, e depois corrija.

**E10.** Compare `BackgroundTasks` com uma tarefa de 5 segundos e 10 requisições simultâneas. Mostre o worker sendo ocupado.

**E11.** Implemente cache-aside com invalidação em **todos** os caminhos de escrita (`POST`, `PATCH`, `DELETE`). Prove que nenhum caminho ficou de fora.

**E12.** 🔴 Reproduza o estouro de cache com 50 threads e resolva com trava. Depois use `SET NX EX` do Redis em vez de `threading.Lock` e explique a diferença.

**E13.** Implemente TTL com jitter e mostre, num histograma, que as chaves expiram espalhadas.

**E14.** Escreva um WebSocket de chat com salas. Prove que uma mensagem numa sala não chega à outra.

**E15.** Implemente autenticação no WebSocket via primeira mensagem, e feche a conexão com código `1008` se o token for inválido.

**E16.** Compare WebSocket e SSE para "notificar quando o relatório ficar pronto". Implemente os dois e argumente qual você escolheria.

In [ ]:
# E1

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

In [ ]:
# E6

In [ ]:
# E7

In [ ]:
# E8

In [ ]:
# E9

In [ ]:
# E10

In [ ]:
# E11

In [ ]:
# E12

In [ ]:
# E13

In [ ]:
# E14

In [ ]:
# E15

In [ ]:
# E16

## 📋 Cola de referência

```python
# ═══ WEBHOOK: receber 🔴 ═══
@app.post("/webhooks/x", status_code=202)
async def receber(req: Request, tarefas: BackgroundTasks,
                  x_assinatura: str | None = Header(default=None)):
    corpo = await req.body()              # 🔴 CRU — reserializar quebra o HMAC
    conferir(corpo, x_timestamp, x_assinatura)
    evento = json.loads(corpo)
    if ja_visto(evento["id"]): return {"repetido": True}
    tarefas.add_task(processar, evento)   # responde rápido
    return {"recebido": True}

esperada = hmac.new(segredo.encode(), ts + b"." + corpo, hashlib.sha256).hexdigest()
hmac.compare_digest(esperada, recebida)   # 🔴 NUNCA `==` (ataque de tempo)

# As 4 verificações: assinatura · timestamp · id repetido · resposta rápida
# Idempotência distribuída:  cache.set(chave, "1", nx=True, ex=86400) is None

# ═══ UPLOAD 🔴 ═══
async def enviar(arquivo: UploadFile = File(...)):   # UploadFile, não bytes
    while pedaco := await arquivo.read(64*1024):     # pedaços, com teto
        ...
    identificar(conteudo[:16])          # magic bytes, não extensão
    Path(arquivo.filename).name         # 🔴 contra path traversal

# ═══ DOWNLOAD ═══
FileResponse(caminho, filename="x.csv")        # arquivo que já existe
StreamingResponse(gerador(), media_type="text/csv",
                  headers={"Content-Disposition": 'attachment; filename="x.csv"'})

# ═══ BACKGROUND 🔴 não é fila ═══
tarefas.add_task(funcao, arg)
# perde tudo no restart · sem retry · mesmo processo
# rápido e descartável → ok   ·   lento ou importante → fila (M10)

# ═══ CACHE ═══
chave = "app:recurso:v1:id"           # namespace:recurso:VERSÃO:id
cache.setex(chave, ttl, json.dumps(dado))
cache.delete(chave)                   # invalide em TODO caminho de escrita
# TTL + invalidação explícita: use os DOIS
# 🔴 estouro: trava (com dupla checagem) · TTL com jitter · recomputar antes

# ═══ WEBSOCKET ═══
@app.websocket("/ws")
async def ws(sock: WebSocket):
    await sock.accept()
    try:
        while True:
            dado = await sock.receive_json()
            await sock.send_json(...)
    except WebSocketDisconnect:
        ...
    finally:
        conexoes.remover(sock)
# transmitir: itere sobre uma CÓPIA da lista
# 🔴 sem cabeçalho Authorization no handshake do navegador
```

## ✅ Checklist de saída

**Webhooks**

- [ ] Entendo que um webhook é um endpoint **público** que muda estado
- [ ] 🔴 **Valido assinatura HMAC com `compare_digest`**
- [ ] Leio o corpo **cru**, sem reserializar
- [ ] Verifico o timestamp contra reenvio
- [ ] 🔴 **Trato o mesmo evento chegando duas vezes**
- [ ] Respondo rápido e processo depois
- [ ] Sei que idempotência em memória não sobrevive ao restart

**Arquivos**

- [ ] Uso `UploadFile`, não `bytes`
- [ ] Leio em pedaços com teto de tamanho
- [ ] 🔴 **Valido magic bytes, não extensão nem `Content-Type`**
- [ ] 🔴 **Sanitizo o nome contra path traversal**
- [ ] Uso `StreamingResponse` para conteúdo gerado

**Assíncrono**

- [ ] Sei que `BackgroundTasks` **não** é fila
- [ ] Sei responder "se esta tarefa se perder, qual o prejuízo?"

**Cache**

- [ ] Minhas chaves têm namespace e **versão**
- [ ] Invalido em todos os caminhos de escrita
- [ ] Uso TTL **e** invalidação explícita
- [ ] 🔴 **Sei o que é estouro de cache e como evitá-lo**
- [ ] Sei o que o `fakeredis` não reproduz

**Tempo real**

- [ ] Sei quando polling basta
- [ ] Removo a conexão no `finally`
- [ ] Transmito iterando sobre uma cópia
- [ ] Sei por que autenticar WebSocket é diferente

---

### ➡️ Próxima aula

**`07_03_Testes.ipynb`** — Tudo o que você construiu nos módulos 06 e 07 precisa continuar funcionando amanhã. pytest, fixtures, `dependency_overrides` e mock de API externa.